# Notebook 4: Decision Trees, CART & Random Forest — A Deep Dive

**BMI 503/511 — Machine Learning: Classification Methods**  
**Instructor:** Pratik Dutta, Ph.D. | Stony Brook University

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duttaprat/BMI_511/blob/main/2026/classification/NB4_DecisionTrees_CART_RandomForest.ipynb)

---

## Learning Objectives
1. Understand the **CART algorithm** (Classification and Regression Trees) in detail
2. Learn how **Gini Impurity** and **Entropy/Information Gain** drive tree splits
3. Visualize how a decision tree grows and how **pruning** controls overfitting
4. Build intuition for **Random Forest** as an ensemble of CART trees
5. Compare **single tree vs. forest** on real clinical data
6. Interpret **feature importance** from both models

**Estimated time: ~30–35 minutes**

---
## Part 1: The CART Algorithm from Scratch

**CART** (Classification and Regression Trees) is the algorithm behind sklearn's `DecisionTreeClassifier`.

### How CART works:
1. At each node, consider every feature and every possible split threshold
2. Choose the split that produces the **purest** child nodes
3. Repeat recursively until a stopping criterion is met

### Two measures of "purity":

| Criterion | Formula | Interpretation |
|-----------|---------|----------------|
| **Gini Impurity** | $G = 1 - \sum_{k} p_k^2$ | Probability of misclassifying a randomly chosen sample |
| **Entropy** | $H = -\sum_{k} p_k \log_2(p_k)$ | Information content / disorder at a node |

Let's compute these by hand first, then let sklearn do it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, learning_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_curve, auc, roc_auc_score)
import warnings
warnings.filterwarnings('ignore')

print("All imports ready! ✅")

### 1.1 Computing Gini and Entropy by Hand

In [ ]:
def gini_impurity(labels):
    """Compute Gini impurity for a set of labels."""
    if len(labels) == 0:
        return 0
    counts = np.bincount(labels)
    probs = counts / len(labels)
    return 1 - np.sum(probs ** 2)

def entropy(labels):
    """Compute entropy for a set of labels."""
    if len(labels) == 0:
        return 0
    counts = np.bincount(labels)
    probs = counts[counts > 0] / len(labels)
    return -np.sum(probs * np.log2(probs))

def information_gain(parent, left_child, right_child, criterion='gini'):
    """Compute information gain from a split."""
    func = gini_impurity if criterion == 'gini' else entropy
    n = len(parent)
    n_left, n_right = len(left_child), len(right_child)
    parent_impurity = func(parent)
    weighted_child = (n_left/n) * func(left_child) + (n_right/n) * func(right_child)
    return parent_impurity - weighted_child

# ---- Example: 10 patients, 6 benign (0), 4 malignant (1) ----
parent = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1])

print("=" * 55)
print("  MANUAL COMPUTATION: Splitting 10 Patients")
print("=" * 55)
print(f"Parent node: 6 Benign, 4 Malignant")
print(f"  Gini:    {gini_impurity(parent):.4f}")
print(f"  Entropy: {entropy(parent):.4f}")

# Suppose feature "tumor size > 3cm" splits them:
left  = np.array([0, 0, 0, 0, 0, 1])  # size <= 3cm: 5 benign, 1 malignant
right = np.array([0, 1, 1, 1])          # size > 3cm:  1 benign, 3 malignant

print(f"\n--- Split: tumor size > 3cm ---")
print(f"  Left child  (≤ 3cm): 5 Benign, 1 Malignant  →  Gini = {gini_impurity(left):.4f}")
print(f"  Right child (> 3cm): 1 Benign, 3 Malignant  →  Gini = {gini_impurity(right):.4f}")
print(f"  Information Gain (Gini):    {information_gain(parent, left, right, 'gini'):.4f}")
print(f"  Information Gain (Entropy): {information_gain(parent, left, right, 'entropy'):.4f}")

# Alternative split
left2  = np.array([0, 0, 0, 1, 1])     # age <= 50: 3 benign, 2 malignant
right2 = np.array([0, 0, 0, 1, 1])     # age > 50:  3 benign, 2 malignant

print(f"\n--- Split: age > 50 ---")
print(f"  Left child  (≤ 50): 3 Benign, 2 Malignant  →  Gini = {gini_impurity(left2):.4f}")
print(f"  Right child (> 50): 3 Benign, 2 Malignant  →  Gini = {gini_impurity(right2):.4f}")
print(f"  Information Gain (Gini):    {information_gain(parent, left2, right2, 'gini'):.4f}")

print(f"\n💡 CART picks the split with HIGHER information gain.")
print(f"   Tumor size ({information_gain(parent, left, right, 'gini'):.4f}) >> Age ({information_gain(parent, left2, right2, 'gini'):.4f})")
print(f"   → Tumor size is a better split!")

### 1.2 Visualizing Gini vs Entropy

How do these impurity measures behave as class proportions change?

In [ ]:
p = np.linspace(0.001, 0.999, 200)
gini_vals = 1 - p**2 - (1-p)**2
entropy_vals = -p * np.log2(p) - (1-p) * np.log2(1-p)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p, gini_vals, color='#3B82F6', lw=2.5, label='Gini Impurity')
ax.plot(p, entropy_vals, color='#EF4444', lw=2.5, label='Entropy')
ax.plot(p, 2 * p * (1-p), color='#10B981', lw=2, linestyle='--', label='Misclassification Error')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Proportion of Class 1 (p)', fontsize=12)
ax.set_ylabel('Impurity', fontsize=12)
ax.set_title('Impurity Measures for Binary Classification', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.annotate('Maximum impurity\n(50/50 split)', xy=(0.5, 0.5), xytext=(0.7, 0.35),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'),
            color='gray')
plt.tight_layout()
plt.show()

print("💡 Both Gini and Entropy are maximized at p=0.5 (most mixed).")
print("   Both are 0 at p=0 or p=1 (pure nodes).")
print("   In practice, Gini and Entropy give very similar trees.")

---
## Part 2: Decision Trees on Real Data

We use the **Wisconsin Breast Cancer** dataset (569 samples, 30 features).

In [ ]:
# Load data
data = load_breast_cancer()
X, y = data.data, data.target  # 0=malignant, 1=benign
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")
print(f"Features: {X.shape[1]}")
print(f"Classes: Malignant={sum(y==0)}, Benign={sum(y==1)}")

### 2.1 Growing a CART Tree — Gini vs Entropy

In [ ]:
# Train two trees: one with Gini, one with Entropy
tree_gini = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
tree_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)

tree_gini.fit(X_train, y_train)
tree_entropy.fit(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(22, 7))

for ax, tree, title in [(axes[0], tree_gini, 'CART with Gini Impurity'),
                         (axes[1], tree_entropy, 'CART with Entropy')]:
    plot_tree(tree, feature_names=feature_names, class_names=['Malignant', 'Benign'],
              filled=True, rounded=True, ax=ax, fontsize=8, proportion=True)
    ax.set_title(title, fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

acc_g = accuracy_score(y_test, tree_gini.predict(X_test))
acc_e = accuracy_score(y_test, tree_entropy.predict(X_test))
print(f"Gini tree accuracy:    {acc_g:.4f}")
print(f"Entropy tree accuracy: {acc_e:.4f}")
print(f"\n💡 The trees may differ slightly in structure but typically give similar accuracy.")

### 2.2 Tree as Text — The Rules a Clinician Can Read

One of the best features of decision trees: you can extract **human-readable rules**.

In [ ]:
# Print tree as text rules
tree_rules = export_text(tree_gini, feature_names=list(feature_names), max_depth=3)
print("DECISION RULES (max_depth=3):")
print("=" * 50)
print(tree_rules)
print("💡 A clinician could follow these rules with just 3 measurements!")
print("   This is why decision trees are popular in clinical decision support.")

### 2.3 The Overfitting Problem — Why Pruning Matters

An unpruned tree will memorize the training data perfectly — but fail on new patients.

In [ ]:
depths = list(range(1, 21))
train_accs, test_accs = [], []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, tree.predict(X_train)))
    test_accs.append(accuracy_score(y_test, tree.predict(X_test)))

# Also unlimited depth
tree_unlimited = DecisionTreeClassifier(random_state=42)
tree_unlimited.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_accs, 'o-', color='#3B82F6', lw=2, label='Training Accuracy')
ax.plot(depths, test_accs, 's-', color='#EF4444', lw=2, label='Test Accuracy')

best_depth = depths[np.argmax(test_accs)]
ax.axvline(best_depth, color='#10B981', linestyle='--', alpha=0.7,
           label=f'Best depth = {best_depth}')

# Shade overfitting zone
ax.axvspan(best_depth + 1, 20, alpha=0.08, color='red')
ax.text(14, 0.935, 'OVERFITTING\nZONE', fontsize=12, color='#EF4444',
        ha='center', fontweight='bold', alpha=0.6)

ax.set_xlabel('Tree Depth (max_depth)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Decision Tree: Training vs Test Accuracy by Depth', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim([0.9, 1.005])
plt.tight_layout()
plt.show()

print(f"\nUnlimited tree: depth={tree_unlimited.get_depth()}, "
      f"{tree_unlimited.get_n_leaves()} leaves")
print(f"  Train accuracy: {accuracy_score(y_train, tree_unlimited.predict(X_train)):.4f}")
print(f"  Test accuracy:  {accuracy_score(y_test, tree_unlimited.predict(X_test)):.4f}")
print(f"\n💡 Training accuracy reaches 1.0 quickly (memorization!).")
print(f"   Test accuracy peaks around depth={best_depth}, then overfits.")
print(f"   This is WHY we prune — simpler trees generalize better.")

### 2.4 Pruning Strategies

CART supports several pruning hyperparameters. Let's explore the key ones.

In [ ]:
# Cost-Complexity Pruning (ccp_alpha) — post-pruning
# Higher alpha = more aggressive pruning
path = tree_unlimited.cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas[:-1]  # exclude trivial (root-only) tree

pruned_train_accs, pruned_test_accs, n_leaves_list = [], [], []
for alpha in alphas:
    tree = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    tree.fit(X_train, y_train)
    pruned_train_accs.append(accuracy_score(y_train, tree.predict(X_train)))
    pruned_test_accs.append(accuracy_score(y_test, tree.predict(X_test)))
    n_leaves_list.append(tree.get_n_leaves())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: accuracy vs alpha
axes[0].plot(alphas, pruned_train_accs, 'o-', color='#3B82F6', markersize=3, label='Train')
axes[0].plot(alphas, pruned_test_accs, 's-', color='#EF4444', markersize=3, label='Test')
best_alpha_idx = np.argmax(pruned_test_accs)
axes[0].axvline(alphas[best_alpha_idx], color='#10B981', linestyle='--',
                label=f'Best α = {alphas[best_alpha_idx]:.4f}')
axes[0].set_xlabel('Cost-Complexity Alpha (ccp_alpha)', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_title('Cost-Complexity Pruning', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Right: number of leaves vs alpha
axes[1].plot(alphas, n_leaves_list, 'o-', color='#8B5CF6', markersize=3)
axes[1].axvline(alphas[best_alpha_idx], color='#10B981', linestyle='--')
axes[1].set_xlabel('Cost-Complexity Alpha (ccp_alpha)', fontsize=11)
axes[1].set_ylabel('Number of Leaves', fontsize=11)
axes[1].set_title('Tree Complexity vs Pruning', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best alpha: {alphas[best_alpha_idx]:.4f}")
print(f"  Leaves: {n_leaves_list[best_alpha_idx]}  |  Test accuracy: {pruned_test_accs[best_alpha_idx]:.4f}")

In [ ]:
# Compare pre-pruning hyperparameters
pruning_configs = {
    'No pruning':           DecisionTreeClassifier(random_state=42),
    'max_depth=3':          DecisionTreeClassifier(max_depth=3, random_state=42),
    'max_depth=5':          DecisionTreeClassifier(max_depth=5, random_state=42),
    'min_samples_leaf=10':  DecisionTreeClassifier(min_samples_leaf=10, random_state=42),
    'min_samples_split=20': DecisionTreeClassifier(min_samples_split=20, random_state=42),
    'max_leaf_nodes=10':    DecisionTreeClassifier(max_leaf_nodes=10, random_state=42),
    f'ccp_alpha={alphas[best_alpha_idx]:.4f}': DecisionTreeClassifier(
        ccp_alpha=alphas[best_alpha_idx], random_state=42),
}

print(f"{'Configuration':<25} {'Depth':>6} {'Leaves':>7} {'Train Acc':>10} {'Test Acc':>10}")
print('=' * 62)

for name, tree in pruning_configs.items():
    tree.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, tree.predict(X_train))
    test_acc = accuracy_score(y_test, tree.predict(X_test))
    print(f"{name:<25} {tree.get_depth():>6} {tree.get_n_leaves():>7} "
          f"{train_acc:>10.4f} {test_acc:>10.4f}")

print("\n💡 Pruning reduces tree complexity and often improves test accuracy.")
print("   The unpruned tree has perfect training accuracy but lower test accuracy.")

---
## Part 3: From One Tree to a Forest

### The problem with a single tree:
- **High variance**: small changes in data → very different tree
- **Greedy**: each split is locally optimal, not globally

### Random Forest solves this with two ideas:
1. **Bagging** (Bootstrap Aggregating): train each tree on a random sample (with replacement)
2. **Feature randomness**: at each split, only consider √p random features

The result: many diverse trees that vote together → lower variance, better generalization.

In [ ]:
# Demonstrate tree instability — train on slightly different data
fig, axes = plt.subplots(1, 3, figsize=(22, 5))

for i, ax in enumerate(axes):
    # Bootstrap sample (sample with replacement)
    np.random.seed(i * 10)
    idx = np.random.choice(len(X_train), size=len(X_train), replace=True)
    X_boot, y_boot = X_train[idx], y_train[idx]

    tree = DecisionTreeClassifier(max_depth=3, random_state=42)
    tree.fit(X_boot, y_boot)
    acc = accuracy_score(y_test, tree.predict(X_test))

    plot_tree(tree, feature_names=feature_names, class_names=['Malig', 'Benign'],
              filled=True, rounded=True, ax=ax, fontsize=7, proportion=True)
    ax.set_title(f'Bootstrap Sample {i+1}\nTest Acc: {acc:.3f}', fontsize=12, fontweight='bold')

plt.suptitle('Same Algorithm, Different Data Samples → Different Trees!',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 This is the HIGH VARIANCE problem of decision trees.")
print("   Random Forest averages over many such trees to reduce variance.")

### 3.1 Random Forest: Effect of Number of Trees

In [ ]:
n_trees_range = [1, 2, 5, 10, 25, 50, 100, 200, 300, 500]
rf_train_accs, rf_test_accs, rf_oob_accs = [], [], []

for n in n_trees_range:
    rf = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42)
    rf.fit(X_train, y_train)
    rf_train_accs.append(accuracy_score(y_train, rf.predict(X_train)))
    rf_test_accs.append(accuracy_score(y_test, rf.predict(X_test)))
    rf_oob_accs.append(rf.oob_score_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_trees_range, rf_train_accs, 'o-', color='#3B82F6', lw=2, label='Training')
ax.plot(n_trees_range, rf_test_accs, 's-', color='#EF4444', lw=2, label='Test')
ax.plot(n_trees_range, rf_oob_accs, '^-', color='#10B981', lw=2, label='OOB (Out-of-Bag)')
ax.set_xlabel('Number of Trees (n_estimators)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Random Forest: Performance vs Number of Trees', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim([0.92, 1.005])
plt.tight_layout()
plt.show()

print("💡 Key observations:")
print("   - Performance improves rapidly up to ~50 trees, then plateaus")
print("   - Unlike single trees, RF does NOT overfit with more trees!")
print("   - OOB score is a free 'validation' estimate (no need for separate val set)")

### 3.2 Peeking Inside: Individual Trees in the Forest

In [ ]:
# Train a forest and look at individual tree predictions
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Get predictions from each individual tree for the test set
individual_preds = np.array([tree.predict(X_test) for tree in rf.estimators_])

# For each test sample, what fraction of trees voted correctly?
correct_votes = (individual_preds == y_test).mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution of vote margins
axes[0].hist(correct_votes, bins=20, color='#0D9488', alpha=0.8, edgecolor='white')
axes[0].axvline(0.5, color='#EF4444', linestyle='--', lw=2, label='Decision boundary (50%)')
axes[0].set_xlabel('Fraction of Trees Voting Correctly', fontsize=11)
axes[0].set_ylabel('Number of Test Samples', fontsize=11)
axes[0].set_title('How Confident is the Forest?', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Right: individual tree accuracies vs forest
tree_accs = [accuracy_score(y_test, tree.predict(X_test)) for tree in rf.estimators_]
axes[1].hist(tree_accs, bins=15, color='#8B5CF6', alpha=0.7, edgecolor='white', label='Individual trees')
forest_acc = accuracy_score(y_test, rf.predict(X_test))
axes[1].axvline(forest_acc, color='#EF4444', lw=2.5, linestyle='-', label=f'Forest: {forest_acc:.3f}')
axes[1].axvline(np.mean(tree_accs), color='#F59E0B', lw=2, linestyle='--',
                label=f'Mean tree: {np.mean(tree_accs):.3f}')
axes[1].set_xlabel('Accuracy', fontsize=11)
axes[1].set_ylabel('Number of Trees', fontsize=11)
axes[1].set_title('Individual Trees vs Forest Ensemble', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Individual tree accuracy: {np.mean(tree_accs):.4f} ± {np.std(tree_accs):.4f}")
print(f"Forest ensemble accuracy: {forest_acc:.4f}")
print(f"\n💡 The forest is BETTER than any individual tree!")
print(f"   This is the 'wisdom of the crowd' effect.")

---
## Part 4: Head-to-Head Comparison

In [ ]:
# Final comparison: single CART vs pruned CART vs Random Forest
final_models = {
    'CART (unpruned)': DecisionTreeClassifier(random_state=42),
    'CART (max_depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'CART (ccp pruned)': DecisionTreeClassifier(ccp_alpha=alphas[best_alpha_idx], random_state=42),
    'Random Forest (100)': RandomForestClassifier(n_estimators=100, random_state=42),
    'Random Forest (500)': RandomForestClassifier(n_estimators=500, random_state=42),
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
colors = ['#64748B', '#3B82F6', '#8B5CF6', '#EF4444', '#10B981']

print(f"{'Model':<25} {'CV Accuracy':>12} {'± Std':>8} {'CV AUC':>10}")
print('=' * 58)

cv_data = []
for (name, model), color in zip(final_models.items(), colors):
    acc_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    auc_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    cv_data.append({'Model': name, 'Accuracy': acc_scores.mean(),
                    'Std': acc_scores.std(), 'AUC': auc_scores.mean(),
                    'scores': acc_scores, 'color': color})
    print(f"{name:<25} {acc_scores.mean():>12.4f} {acc_scores.std():>8.4f} {auc_scores.mean():>10.4f}")

# Box plot
fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot([d['scores'] for d in cv_data],
                labels=[d['Model'] for d in cv_data],
                patch_artist=True, widths=0.6)
for patch, d in zip(bp['boxes'], cv_data):
    patch.set_facecolor(d['color'])
    patch.set_alpha(0.6)
ax.set_ylabel('10-Fold CV Accuracy', fontsize=12)
ax.set_title('CART vs Random Forest — Cross-Validation', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

### 4.1 Feature Importance Comparison

In [ ]:
# Train final models on full training set
cart = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)

# Get importances
cart_imp = pd.Series(cart.feature_importances_, index=feature_names).nlargest(10)
rf_imp = pd.Series(rf.feature_importances_, index=feature_names).nlargest(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cart_imp.sort_values().plot(kind='barh', color='#3B82F6', ax=axes[0], alpha=0.8)
axes[0].set_title('Single CART — Feature Importance', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Importance')

rf_imp.sort_values().plot(kind='barh', color='#EF4444', ax=axes[1], alpha=0.8)
axes[1].set_title('Random Forest — Feature Importance', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

# Overlap
cart_top = set(cart_imp.index)
rf_top = set(rf_imp.index)
print(f"\nTop-10 overlap: {len(cart_top & rf_top)} features in common")
print(f"Only in CART:         {cart_top - rf_top}")
print(f"Only in RF:           {rf_top - cart_top}")
print(f"\n💡 Single tree importance is 'spiky' (a few features dominate).")
print(f"   RF importance is smoother — more features contribute.")
print(f"   RF importance is generally more reliable for biomarker discovery.")

### 4.2 Learning Curves — Do We Need More Data?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model), color in [
    (axes[0], ('CART (depth=5)', DecisionTreeClassifier(max_depth=5, random_state=42)), '#3B82F6'),
    (axes[1], ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)), '#EF4444'),
]:
    train_sizes, train_scores, test_scores = learning_curve(
        model, X_train, y_train, cv=5,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy', random_state=42
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    test_mean = test_scores.mean(axis=1)
    test_std = test_scores.std(axis=1)

    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                    alpha=0.1, color=color)
    ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std,
                    alpha=0.1, color='#10B981')
    ax.plot(train_sizes, train_mean, 'o-', color=color, label='Training')
    ax.plot(train_sizes, test_mean, 's-', color='#10B981', label='Validation')
    ax.set_xlabel('Training Set Size', fontsize=11)
    ax.set_ylabel('Accuracy', fontsize=11)
    ax.set_title(f'Learning Curve: {name}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(alpha=0.3)
    ax.set_ylim([0.88, 1.01])

plt.tight_layout()
plt.show()

print("💡 If the gap between train and validation is large → overfitting.")
print("   If both curves are still rising → more data would help.")
print("   RF typically has a smaller gap than a single tree.")

---
## Summary

| Concept | Single CART | Random Forest |
|---------|------------|---------------|
| **Interpretability** | ★★★★★ (read the rules!) | ★★★ (feature importance) |
| **Overfitting risk** | High (needs pruning) | Low (bagging helps) |
| **Variance** | High | Low |
| **Speed** | Very fast | Slower (100s of trees) |
| **Feature importance** | Unstable | Stable & reliable |
| **Best for** | Explainable clinical rules | Best accuracy on tabular data |

### When to use what:
- **Need to explain decisions to a doctor?** → Single pruned CART
- **Need best predictive accuracy?** → Random Forest
- **Small dataset, worried about overfitting?** → Random Forest with few trees

---
## ✅ Exercises for Students

1. **Gini vs Entropy**: Train an unpruned tree with each criterion. Compare depth, number of leaves, and test accuracy. Are the top splits the same?

2. **Feature subsampling**: In Random Forest, change `max_features` from `'sqrt'` (default) to `'log2'`, `0.5`, and `None` (all features). How does it affect accuracy and training time?

3. **Out-of-Bag (OOB) estimation**: OOB score is a free validation metric. Compare OOB score to 5-fold CV accuracy for forests with 50, 100, and 200 trees.

4. **Clinical exercise**: Using the pruned CART tree rules from Section 2.2, create a simple flowchart that a clinician could follow. How many measurements would they need?

5. **Bonus — Extra Trees**: Try `ExtraTreesClassifier` from sklearn. How does it differ from Random Forest in both accuracy and training speed?

```python
# Hint for Exercise 5:
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(n_estimators=100, random_state=42)
```